METHODS WE WILL USE

1)Preprocessing chain (normalization → log → Pareto scaling) 

2)PLS regression in the body_score + LOO-CV

3)PLS-DA στο body_class + LOO-CV (feasibility check )

4)VIP scores 

5)Permutation test στο Q² (mandatory with n=17)

6)MLflow logging 




data from Skogerson, K., Runnebaum, R., Wohlgemuth, G., De Ropp, J., Heymann, H., & Fiehn, O. (2009). Comparison of Gas Chromatography-Coupled Time-of-Flight Mass Spectrometry and1 H Nuclear Magnetic Resonance Spectroscopy Metabolite Identification in White Wines from a Sensory Study Investigating Wine Body. Journal of Agricultural and Food Chemistry, 57(15), 6899–6907. https://doi.org/10.1021/jf9019322



In [ ]:
import pandas as pd

ann = pd.read_excel("/home/nasia/wine-innovation-engine/data/processed/summary_421340.xls", sheet_name="Annotation Data")

ann.head(30)

In [ ]:
import pandas as pd 
import numpy as np


#X needs to be a pandas dataframe with 17 WINES and 109 metabolites ,we have GC-TOF-MS peak intensity ,mean of the 6 replicates for each wine and metabolite,the data ->normalised peak intensity 

def check_zeros(X:pd.DataFrame)->None:
    n_zero_total=(X==0).sum().sum() #sum over columns and total
    n_nan_total=X.isna().sum().sum() #sum over columns and total
    print(f"Total number of zeros in the dataset: {n_zero_total}")
    print(f"Total number of NaN values in the dataset: {n_nan_total}")
    n_cells=X.shape[0]*X.shape[1] #total number of cells in the dataset
    print(f"Percentage of zeros in the dataset: {n_zero_total/n_cells*100:.2f}%")
    print(f"Percentage of NaN values in the dataset: {n_nan_total/n_cells*100:.2f}%")


    zeros_per_col=(X==0).sum() #sum over rows for each column
    cols_with_zeros=zeros_per_col[zeros_per_col>0].sort_values(ascending=False) #the head will shows the worst metabolites with the most zeros
    print(f"Number of columns with zeros: {len(cols_with_zeros)}")  
    print(f"head of the metabolites with the most zeros:\n{cols_with_zeros.head()}")

    print(f"columns with at least one zero: {len(cols_with_zeros)}/{X.shape[1]}")
    
    if len(cols_with_zeros)>0:
        print(cols_with_zeros.head(10))


def log_transform(X:pd.DataFrame,offset:float=1)->pd.DataFrame:
    #log transform the data, we will use log10(x+1) to avoid log(0)
       return np.log10(X+offset)

def pareto_scale(X:pd.DataFrame)->pd.DataFrame:
    #pareto scaling, we will use the standard deviation of each metabolite to scale the data
    return (X-X.mean())/np.sqrt(X.std())





In [ ]:
#classID WITH body score and body class from the paper

xwalk = {421745:('CH01',2.57,'high'),421770:('CH02',2.85,'high'),421795:('CH03',3.21,'high'),
 421820:('CH04',2.55,'high'),421845:('CH05',2.91,'high'),421870:('CH06',2.46,'medium'),
 421545:('PG01',2.13,'low'),421570:('PG02',2.24,'low'),421595:('R01',2.44,'medium'),
 421620:('R02',2.63,'high'),421645:('SB01',2.16,'low'),421670:('SB02',1.97,'low'),
 421695:('SB03',2.33,'medium'),421720:('SB04',2.12,'low'),421495:('V01',2.42,'medium'),
 421520:('V02',2.32,'medium'),421895:('WW01',2.22,'low')}

#orient ='index' means that the keys of the dictionary will be used as the index of the DataFrame, and the values will be used as the data for the columns. The columns parameter specifies the names of the columns in the DataFrame.  

meta=pd.DataFrame.from_dict(xwalk,orient='index',columns=['wine','body_score','body_class'])
meta.index.name='ClassID'

feature_cols = [
    c for c in ann.columns
    if c not in ('SampleID', 'ClassID')
    and not c.strip().replace('.', '', 1).isdigit()   # removes unnamed BinBase feature (π.χ. "226091.0")
] #it takes the metabolites columns only, 109 metabolites
X_wine = ann.groupby('ClassID')[feature_cols].mean()  

data = meta.join(X_wine).reset_index()
data.shape  

X=data.set_index('ClassID')[feature_cols]  #17 wines and 109 metabolites
check_zeros(X)
X_log=log_transform(X)
X_scaled=pareto_scale(X_log)




In [ ]:
data.shape

In [ ]:
X_scaled.head()

In [ ]:
meta

WE will do PLS because we have a higher number of deegrees of freedom and we need to find the latent components ,we also need to do LOO-CV and not the regular 5-fold because we have limited samples

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import r2_score

def loo_cv_q2(X:pd.DataFrame,y:pd.Series,n_components:int)->tuple[float,np.ndarray]:

    loo=LeaveOneOut()
    y_pred=np.zeros(len(y))#we need it to save the 17 predictions for each wine, we will use it to calculate the q2 score
    for train_index,test_index in loo.split(X):#loo.split is a generator that yields the train and test indices for each iteration of the leave-one-out cross-validation
        X_train,X_test=X.iloc[train_index],X.iloc[test_index]
        y_train,y_test=y.iloc[train_index],y.iloc[test_index]
        pls=PLSRegression(n_components=n_components)
        pls.fit(X_train,y_train)
        y_pred[test_index]=pls.predict(X_test).ravel() #ravel() flattens the array to 1D


    q2=r2_score(y,y_pred)#it needs the prediction and the real values to calculate the q2 score
    return q2,y_pred

def select_n_components(X: pd.DataFrame, y: pd.Series, max_components: int = 6) -> pd.DataFrame:
    #it tries different components from 1 to max_components and returns a dataframe with the q2 score for each number of components
    results = []
    for n in range(1, max_components + 1):
        q2, _ = loo_cv_q2(X, y, n)
        results.append((n, q2))
    return pd.DataFrame(results, columns=["n_components", "Q2"])

y=data.set_index('ClassID')['body_score']
scores_table=select_n_components(X_scaled,y,max_components=6)
print(scores_table)
                    

In [ ]:
final_pls=PLSRegression(n_components=1) #we pick one because the first was the best
final_pls.fit(X_scaled,y)

In [ ]:
y

In [ ]:
#R^2 Y calibration (in-sample ,always optimistic)

y_pred_calibration=final_pls.predict(X_scaled).ravel() #ravel() flattens the array to 1D

y_pred_calibration

r2y_calibration = r2_score(y, y_pred_calibration)


#it will gives a scalar (the best component) and the predictions for each wine, we will use it to calculate the q2 score
best_q2, y_pred_loo = loo_cv_q2(X_scaled, y, n_components=1)

print(f"R²Y (calibration, in-sample): {r2y_calibration:.3f}")
print(f"Q²  (LOO-CV, out-of-sample):  {best_q2:.3f}")
print(f"Overfitting gap (R²Y - Q²):   {r2y_calibration - best_q2:.3f}")

We have a lower overfitting gap and a lower best q^2 because of missing many samples 

PLSA-DA binary low vs high+medium it needs 0 or 1 

In [ ]:
y_pred_calibration

In [ ]:

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score

#we will categorize the wines based on their body score, into low ->0 and medium+high->1
y_da=(meta['body_class']!='low').astype(int) #it makes a boolean and asks its not low (true or false) ,as type int it will be 1 for true and 0 for false, so we have a binary classification problem

y_da=y_da.loc[X_scaled.index] #we need to make sure that the index of y_da is the same as X_scaled, so we will use the index of X_scaled to filter y_da


print("Κατανομή κλάσεων:")
y_da.value_counts() #it will show us how many wines are in each class, we have 7 low and 10 medium+high






In [ ]:
majority_class_count=y_da.value_counts().max() #it will show us how many wines are in the majority class, we have 10 medium+high
total=len(y_da) #total number of wines

majority_baseline=majority_class_count/total #it will show us the accuracy of the majority class baseline
print(f"Majority classifier baseline accuracy: {majority_baseline:.3f}")


Majority baseline (0.647): if we always say '1' without looking at any metabolite, we get 64.7% accuracy for free. PLS-DA must overcome this to be said to learn something.

In [ ]:
#LOO-CV for selecting n_components

def loo_cv_da(X:pd.DataFrame,y:pd.Series,n_components:int):
    loo=LeaveOneOut()
    y_pred_class=np.zeros(len(y),dtype=int) #we need it to save the 17 predictions for each wine, we will use it to calculate the accuracy score

    for train_idx,test_idx in loo.split(X):
        X_train,X_test=X.iloc[train_idx],X.iloc[test_idx]
        y_train,y_test=y.iloc[train_idx],y.iloc[test_idx]

        pls=PLSRegression(n_components=n_components)
        pls.fit(X_train,y_train)

        y_hat=pls.predict(X_test).ravel()[0] #ravel() flattens the array to 1D, we need the first element because it returns an array with one element
        y_pred_class[test_idx]=1 if y_hat>=0.5 else 0 #if the prediction is greater than or equal to 0.5 we classify it as 1 (medium+high) else 0 (low)

    acc=accuracy_score(y,y_pred_class) #it needs the prediction and the real values to calculate the accuracy score
    bal_acc=balanced_accuracy_score(y,y_pred_class) #it needs the prediction and the real values to calculate the balanced accuracy score
    return acc,bal_acc,y_pred_class

print("\nComponent selection:")
for n in range(1, 5):
    acc, bal_acc, _ = loo_cv_da(X_scaled, y_da, n)
    print(f"  n_components={n}  acc={acc:.3f}  balanced_acc={bal_acc:.3f}")

In [ ]:
best_n_da = 3
final_acc, final_bal_acc, y_pred_final = loo_cv_da(X_scaled, y_da, n_components=best_n_da)

print(f"Επιλεγμένο n_components: {best_n_da}")
print(f"LOO Accuracy:          {final_acc:.3f}")
print(f"LOO Balanced Accuracy: {final_bal_acc:.3f}")
print(f"Majority baseline:     {y_da.value_counts().max()/len(y_da):.3f}")


#balanced accuracy is the average of recall obtained on each class. It is used when the classes are imbalanced, as it gives equal weight to both classes. In our case, we have 7 low and 10 medium+high, so the balanced accuracy is more informative than the regular accuracy.



In [ ]:
y

In [ ]:
#the accuracies are very high so first we will do a permutation test to see if the model is significant, we will permute the labels 1000 times and see how many times we get an accuracy higher than the original accuracy, if it is less than 5% then we can say that the model is significant
#we will not use the scipy permutation test because it expects a simple statistic function that does 17 fits and splits and also does permutation on x and not at y 
#permutation test asks 'How well can this pipeline, with these X's and this n=17, 'find' patterns that don't exist?'
def permutation_test_da(X:pd.DataFrame,y:pd.Series,n_components:int=1000,n_permutations:int=1000,random_state:int=42):

    rng=np.random.default_rng(random_state) #it gives us a random number generator that we can use to permute the labels, we will use it to shuffle the labels 1000 times and see how many times we get an accuracy higher than the original accuracy

    _,real_bal_acc,_=loo_cv_da(X,y,n_components) #it gives us the real balanced accuracy of the model,loo_cv_da is a function that does leave-one-out cross-validation and returns the accuracy and balanced accuracy of the model, we will use it to get the real balanced accuracy of the model

    #null distribution
    null_bal_accs=[] #it will save the balanced accuracies of the permuted labels, we will use it to create a null distribution of balanced accuracies, we will use it to see how many times we get an accuracy higher than the original accuracy
    for i in range(n_permutations):
        y_shuffled=y.copy()
        y_shuffled.values[:]=rng.permutation(y_shuffled.values) #it shuffles the labels, we will use it to create a null distribution of balanced accuracies, we will use it to see how many times we get an accuracy higher than the original accuracy
        _, perm_bal_acc, _ = loo_cv_da(X, y_shuffled, n_components)
        null_bal_accs.append(perm_bal_acc)

    #this is a bollean array that tells us how many times we get an accuracy higher than the original accuracy, we will use it to calculate the p-value
    null_bal_accs = np.array(null_bal_accs)  
    p_value=(null_bal_accs >= real_bal_acc).mean()#it gives us the p-value of the permutation test, we will use it to see if the model is significant, if it is less than 0.05 then we can say that the model is significa

 
        
    print(f"Real balanced accuracy:   {real_bal_acc:.3f}")
    print(f"Null mean ± std:          {null_bal_accs.mean():.3f} ± {null_bal_accs.std():.3f}")
    print(f"Permutation p-value:      {p_value:.4f}")
    
    return real_bal_acc, null_bal_accs, p_value

real_bal_acc, null_dist, p_val = permutation_test_da(
    X_scaled, y_da, n_components=3, n_permutations=1000
)
   

